In [ ]:
#| default_exp edit_interactive

## Edit-interactive plan execution

Run a tiny Lisette agent loop against one notebook at a time. The inner agent receives the user plan plus a single compact notebook view, then can mutate only that notebook through scoped tools.

Sometimes a user wants an agent to carry out a bounded notebook edit rather than manually choose each `write_nb` or `update_cell` call. This notebook builds that inner edit loop: one notebook, one plan, a small set of notebook-aware tools, and a final diff.

The edit loop is for bounded delegation, not open-ended repository work. Its job is to give an inner agent a stable notebook view, a small tool belt, and revision-aware feedback so a single notebook can be edited and reviewed without exposing raw notebook JSON.

```python
execute_plan("nbs/02_write.ipynb", "Add an example after the write_nb docs", max_steps=4)
```

### Production contract

The edit-interactive loop is experimental. It stays out of the production core unless it has focused contract tests for bounded scope, stable notebook views, deterministic tool results, final diffs, and clear failure behavior when an inner edit cannot be completed.


In [ ]:
from contextlib import redirect_stdout
from io import StringIO
import nbskill.edit_interactive as ei
from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import read_nb as _read_tmp_nb, write_nb as _write_tmp_nb
from nbskill.edit_interactive import notebook_view as _example_notebook_view
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook

In [ ]:
with write_demo_notebook("08_edit_interactive_example.ipynb") as path:
    _write_tmp_nb(new_nb([
        mk_cell("#| default_exp demo"),
        mk_cell("#| export\ndef answer():\n    return 42"),
    ]), path)
    print("\n".join(_example_notebook_view(path).splitlines()[:6]))

In [ ]:
#| export
import difflib
import os
from contextlib import redirect_stderr, redirect_stdout
from dataclasses import dataclass, field
from io import StringIO
from pathlib import Path

from fastcore.nbio import mk_cell
from fastcore.nbio import read_nb
from fastcore.nbio import write_nb
from nbdev.doclinks import nbdev_export as _run_nb_export

from nbskill.execute import exec_nb
from nbskill.foundation import (
    cell_source, clear_outputs, exported_py_path, parse_one_cell,
    stamp_export_metadata, stamp_notebook_metadata, validate_code_cells,
)
from nbskill.parallel import notebook_locks
from nbskill.review import diff_nb

### The inner-agent contract

The system prompt is intentionally narrow. The inner agent edits exactly one notebook, uses only the provided tools, prefers stable cell ids, and stops with a summary when the plan is done.

In [ ]:
#| export
EDIT_INTERACTIVE_SYSTEM = """You are nbskill edit-interactive. You edit exactly one notebook.

You receive a plan and one notebook view. Use only the provided notebook tools.
Prefer id-based edits when the view gives one. Keep changes small and aligned
with the plan. Run cells when that is needed to verify the work. Stop when the
plan is complete. Your final message must summarize what you did and what you
could not do.
"""

In [ ]:
#| export
def capture_call_text(func, **kwargs):
    "Run `func` and return captured stdout/stderr, or the return value."
    out, err = StringIO(), StringIO()
    with redirect_stdout(out), redirect_stderr(err):
        result = func(**kwargs)
    chunks = []
    if out.getvalue(): chunks.append(out.getvalue().rstrip())
    if err.getvalue(): chunks.append(err.getvalue().rstrip())
    if result is not None and not chunks: chunks.append(str(result))
    return "\n".join(chunk for chunk in chunks if chunk)

### A stable notebook view

The edit loop needs a text representation that is compact enough for a model but precise enough for safe edits. `notebook_view` includes ids, cell types, and full cell source.

In [ ]:
#| export
def notebook_view(path, revision=0):
    "Render one notebook as a compact, stable text view."
    path = Path(path)
    with notebook_locks(path):
        nb = read_nb(path)
        lines = [f"Notebook: {path}", f"Revision: {revision}", ""]
        for idx, cell in enumerate(nb.cells):
            lines.append(f"CELL {idx} id={cell.id} type={cell.cell_type}")
            lines.append("<<<SOURCE")
            lines.append(cell_source(cell).rstrip())
            lines.append("SOURCE")
            lines.append("")
        return "\n".join(lines).rstrip() + "\n"

### Session state

`EditSession` tracks the notebook path, revision, timeout, and operation logs. The revision count makes it clear which tool calls changed the notebook during a plan.

In [ ]:
#| export
@dataclass
class EditSession:
    "Mutable state for one edit-interactive notebook run."
    path: Path
    timeout: int = 30
    revision: int = 0
    log: list[str] = field(default_factory=list)
    tool_log: list[str] = field(default_factory=list)
    history: list[dict] = field(default_factory=list)
    chat: object | None = None
    notebook_msg_idx: int | None = None

    def refresh_view(self):
        "Replace the notebook-view message in the Lisette history."
        if self.chat is None or self.notebook_msg_idx is None: return
        msg = self.chat.hist[self.notebook_msg_idx]
        view = notebook_view(self.path, self.revision)
        if isinstance(msg, dict): msg["content"] = view
        else: msg.content = view

    def record(self, message):
        "Append an operation to the session log."
        self.log.append(f"r{self.revision}: {message}")

    def record_tool(self, name, detail=""):
        "Append one tool use to the session history."
        suffix = f"({detail})" if detail else "()"
        self.tool_log.append(f"r{self.revision}: {name}{suffix}")
        item = {"revision": self.revision, "tool": name}
        if detail: item["detail"] = detail
        self.history.append(item)

In [ ]:
#| export
def _export_notebook(nb, nb_path):
    py_path = exported_py_path(nb_path, nb)
    if py_path is None: return None
    _run_nb_export(path=str(nb_path))
    if py_path.exists():
        stamp_export_metadata(nb, py_path)
        write_nb(nb, nb_path)
    return py_path


def _save_notebook(nb, path):
    with notebook_locks(path):
        stamp_notebook_metadata(nb)
        write_nb(nb, path)
        _export_notebook(nb, path)

In [ ]:
#| export
def _none_if_blank(value):
    if value is None: return None
    value = str(value)
    return None if value.strip().lower() in {"", "none", "null"} else value

In [ ]:
#| export
def _edit_find_cell_by_id(cells, cell_id):
    matches = [(idx, cell) for idx, cell in enumerate(cells) if getattr(cell, "id", None) == str(cell_id)]
    if len(matches) == 1: return matches[0]
    if not matches: raise ValueError(f"No cell has id {cell_id!r}")
    raise ValueError(f"Multiple cells have id {cell_id!r}")

In [ ]:
#| export
def _source_diff(before, after, label):
    diff = difflib.unified_diff(
        before.splitlines(True), after.splitlines(True),
        fromfile=f"{label}:before", tofile=f"{label}:after",
    )
    text = "".join(diff).strip()
    return text or "No source changes"

In [ ]:
#| export
def _finish_write(session, nb, message, diff):
    _save_notebook(nb, session.path)
    session.revision += 1
    session.record(message)
    session.refresh_view()
    return f"{message}\nrevision={session.revision}\n\n{diff}"

In [ ]:
#| export
def _validate_cell(cell):
    validate_code_cells([cell])
    return cell

### The editing tool belt

The inner loop only gets four notebook operations: add, edit, run through a cell, and remove. Each operation validates inputs, refreshes the notebook view, and records a diff-like message for the final report.

In [ ]:
#| export
def make_edit_tools(session):
    "Create notebook-scoped tools for one edit-interactive session."

    def add_cell(
        after_id: str | None = None,  # Cell id to insert after; blank/None appends
        content: str = "",  # New cell source; may start with %%markdown, %%code, or %%raw
    ) -> str:
        "Add one cell to the current notebook."
        with notebook_locks(session.path):
            nb = read_nb(session.path)
            new_cell = _validate_cell(parse_one_cell(content, "code"))
            anchor = _none_if_blank(after_id)
            target = len(nb.cells)
            if anchor is not None:
                idx, _ = _edit_find_cell_by_id(nb.cells, anchor)
                target = idx + 1
            session.record_tool("add_cell", f"after_id={anchor!r}")
            nb.cells.insert(target, new_cell)
            src = cell_source(new_cell)
            where = f"after id={anchor}" if anchor is not None else "at end"
            msg = f"Added cell id={new_cell.id} {where}"
            return _finish_write(session, nb, msg, _source_diff("", src, f"cell {new_cell.id}"))

    def edit_cell(
        id: str,  # Cell id to edit
        new_src: str,  # Replacement source, or substring replacement when old_src is set
        old_src: str | None = None,  # Optional exact text to replace inside the cell
    ) -> str:
        "Edit one cell in the current notebook."
        with notebook_locks(session.path):
            nb = read_nb(session.path)
            idx, cell = _edit_find_cell_by_id(nb.cells, id)
            before = cell_source(cell)
            old_src = _none_if_blank(old_src)
            session.record_tool(
                "edit_cell",
                f"id={id!r}, old_src={'yes' if old_src is not None else 'no'}",
            )
            if old_src is None:
                new_cell = _validate_cell(parse_one_cell(new_src, cell.cell_type))
                clear_outputs(new_cell)
                new_cell.id = cell.id
                nb.cells[idx] = new_cell
                after = cell_source(new_cell)
                mode = "cell"
            else:
                if old_src not in before: raise ValueError(f"old_src was not found in id={id}")
                after = before.replace(old_src, new_src, 1)
                if cell.cell_type == "code": _validate_cell(mk_cell(after, cell_type="code"))
                cell.source = after
                clear_outputs(cell)
                mode = "text"
            msg = f"Edited {mode} id={id}"
            return _finish_write(session, nb, msg, _source_diff(before, after, f"cell {id}"))

    def run_cell(
        id: str,  # Cell id to execute through
    ) -> str:
        "Run notebook cells up to and including this cell id."
        with notebook_locks(session.path):
            _edit_find_cell_by_id(read_nb(session.path).cells, id)
            session.record_tool("run_cell", f"id={id!r}")
            text = capture_call_text(
                exec_nb, path=str(session.path), dest=str(session.path), up2id=str(id),
                timeout=session.timeout, show_output=True, verbose=False,
            )
            status = "error" if "Traceback (most recent call last)" in text else "ok"
            session.record(f"Ran cells through id={id} ({status})")
            return f"status={status}\n{text}" if text else f"Executed through id={id} ({status})"

    def remove_cell(
        id: str,  # Cell id to remove
    ) -> str:
        "Remove one cell from the current notebook."
        with notebook_locks(session.path):
            nb = read_nb(session.path)
            idx, cell = _edit_find_cell_by_id(nb.cells, id)
            session.record_tool("remove_cell", f"id={id!r}")
            before = cell_source(cell)
            del nb.cells[idx]
            msg = f"Removed cell id={id}"
            return _finish_write(session, nb, msg, _source_diff(before, "", f"cell {id}"))

    return [add_cell, edit_cell, run_cell, remove_cell]

In [ ]:
#| export
def make_chat(model, tools, hist, system_prompt=EDIT_INTERACTIVE_SYSTEM):
    "Create the Lisette chat object for edit-interactive."
    from lisette import Chat
    return Chat(model, sp=system_prompt, tools=tools, hist=hist, stream=str(model).startswith("chatgpt/"))

In [ ]:
#| export
def response_text(response):
    "Extract readable text from a Lisette response or response list."
    if isinstance(response, list) and response: response = response[-1]
    try:
        message = response.choices[0].message
        content = message.content
    except (AttributeError, IndexError, TypeError):
        return "" if response is None else str(response)
    if isinstance(content, list):
        return "".join(str(item.get("text", item)) if isinstance(item, dict) else str(item) for item in content)
    return "" if content is None else str(content)


def plan_result_text(result):
    "Return the human-readable text for an execute_plan result."
    if not isinstance(result, dict): return "" if result is None else str(result)
    if result.get("text"): return str(result["text"])
    sections = [
        "edit-interactive complete",
        "",
        "Final response:",
        str(result.get("summary") or "(no final response)"),
        "",
        "Tools used:",
        "\n".join(item.get("detail", item.get("tool", "")) for item in result.get("history", [])) or "(no tool calls)",
    ]
    return "\n".join(sections).rstrip()

In [ ]:
#| export
def final_diff(path):
    "Return a nbdev code-cell diff, or a clear unavailable message."
    try:
        with notebook_locks(path):
            return capture_call_text(diff_nb, path=str(path))
    except BaseException as exc:
        detail = str(exc)
        if "Could not find notebook" in detail or "No git repository found" in detail:
            return (
                "Code-cell diff against HEAD is unavailable because this notebook has no git baseline. "
                "This is expected for new or untracked notebooks."
            )
        return f"Code-cell diff unavailable: {type(exc).__name__}: {exc}"

### Running a plan

`execute_plan` packages the user's plan, the current notebook view, and the notebook tools into a Lisette chat. The result includes the final answer, tool log, operation log, revision, and code-cell diff.

In [ ]:
#| export
def execute_plan(
    notebook: str,  # Path to the one notebook the inner agent may edit
    plan: str,  # Plan for the inner agent to execute
    model: str | None = None,  # Lisette/LiteLLM model; defaults via NBSKILL_AGENT then a Codex model
    max_steps: int = 20,  # Maximum Lisette tool-loop steps
    timeout: int = 30,  # Per-cell execution timeout for run_cell
    dry_run: bool = False,  # Return proposal context without invoking the inner agent
) -> dict:
    "Execute `plan` against one notebook using a Lisette edit-interactive loop."
    path = Path(notebook)
    if not path.exists(): raise ValueError(f"Notebook does not exist: {notebook}")
    model = model or os.environ.get("NBSKILL_AGENT") or "chatgpt/gpt-5.4-mini"
    initial_view = notebook_view(path, revision=0)
    if dry_run:
        text = "\n".join([
            "edit-interactive dry run",
            "",
            f"Notebook: {path}",
            f"Model: {model}",
            f"Max steps: {max_steps}",
            "Plan:",
            plan,
            "",
            "Initial notebook view:",
            initial_view,
        ]).rstrip()
        return {"summary": "Dry run only; no agent was invoked and no notebook edits were made.", "history": [], "text": text}
    session = EditSession(path=path, timeout=timeout)
    hist = [
        {"role": "user", "content": f"Plan:\n{plan}"},
        {"role": "user", "content": initial_view},
    ]
    tools = make_edit_tools(session)
    chat = make_chat(model, tools=tools, hist=hist)
    session.chat = chat
    session.notebook_msg_idx = len(chat.hist) - 1
    session.refresh_view()
    try:
        result = chat(
            "Execute the plan using only the notebook tools. Stop when the plan is complete. "
            "Your final answer must say what you did and what you could not do.",
            max_steps=max_steps, return_all=True,
        )
    except BaseException as exc:
        result = f"edit-interactive failed: {type(exc).__name__}: {exc}"
    if not isinstance(result, (list, str, bytes, dict)) and hasattr(result, "__next__"):
        result = list(result)
    summary = response_text(result).strip() or "(no final response)"
    diff = final_diff(path).strip()
    text = "\n".join([
        "edit-interactive complete",
        "",
        "Final response:",
        summary,
        "",
        "Tools used:",
        "\n".join(session.tool_log) if session.tool_log else "(no tool calls)",
        "",
        "Operation log:",
        "\n".join(session.log) if session.log else "(no notebook operations)",
        "",
        f"Final revision: {session.revision}",
        "",
        "Notebook diff:",
        diff,
    ]).rstrip()
    return {
        "summary": summary,
        "history": session.history,
        "text": text,
        "model": model,
        "notebook": str(path),
        "revision": session.revision,
        "operations": list(session.log),
        "diff": diff,
    }

In [ ]:
#| export
def _split_notebooks(notebooks):
    if notebooks is None: return []
    if isinstance(notebooks, (list, tuple, set)): return [str(item) for item in notebooks if str(item).strip()]
    return [item.strip() for item in str(notebooks).split(",") if item.strip()]

In [ ]:
#| export
def execute_project_plan(
    plan: str,  # Broad project plan to split into notebook-scoped executions
    notebooks: str | None = None,  # Comma-separated notebooks to target
    model: str | None = None,  # Lisette/LiteLLM model
    max_steps: int = 20,  # Maximum steps per notebook
    timeout: int = 30,  # Per-cell execution timeout
    dry_run: bool = True,  # Return per-notebook proposals without mutation by default
) -> str:
    "Coordinate a broad plan as notebook-scoped execute_plan calls."
    targets = _split_notebooks(notebooks)
    if not targets: raise ValueError("Pass one or more notebooks to execute_project_plan.")
    seen = set()
    repeated = sorted({path for path in targets if path in seen or seen.add(path)})
    if repeated: raise ValueError(f"Duplicate notebook target(s): {', '.join(repeated)}")
    chunks = ["project execute_plan coordinator", "", f"Dry run: {dry_run}", f"Targets: {len(targets)}"]
    for notebook in targets:
        subplan = f"{plan}\n\nScope: edit only {notebook}."
        result = execute_plan(
            notebook=notebook, plan=subplan, model=model, max_steps=max_steps,
            timeout=timeout, dry_run=dry_run,
        )
        chunks.extend(["", f"## {notebook}", plan_result_text(result)])
    return "\n".join(chunks).rstrip()

In [ ]:
with write_demo_notebook("08_edit_view.ipynb") as path:
    _write_tmp_nb(new_nb([
        mk_cell("#| default_exp sample"),
        mk_cell("#| export\ndef public():\n    return 1"),
        mk_cell("assert public() == 1"),
    ]), path)
    view = ei.notebook_view(path, revision=3)
    assert "Revision: 3" in view
    assert "type=code" in view
    assert "type=code" in view
    assert "public()" in view

In [ ]:
with write_demo_notebook("08_edit_tools.ipynb") as path:
    first = mk_cell("#| default_exp sample")
    second = mk_cell("x = 1")
    _write_tmp_nb(new_nb([first, second]), path)
    session = ei.EditSession(path=path)
    class FakeChat:
        def __init__(self): self.hist = [{"role": "user", "content": "plan"}, {"role": "user", "content": ei.notebook_view(path)}]
    session.chat = FakeChat()
    session.notebook_msg_idx = 1
    add_cell, edit_cell, run_cell, remove_cell = ei.make_edit_tools(session)
    added = add_cell(second.id, "y = 2")
    nb = _read_tmp_nb(path)
    assert "Added cell" in added
    assert len(nb.cells) == 3
    assert "y = 2" in session.chat.hist[1]["content"]
    new_id = nb.cells[-1].id
    edited = edit_cell(new_id, "y = 3", )
    assert "Edited cell" in edited
    nb = _read_tmp_nb(path)
    assert nb.cells[-1].source == "y = 3"
    removed = remove_cell(new_id, )
    assert "Removed cell" in removed
    assert len(_read_tmp_nb(path).cells) == 2
    assert session.revision == 3

In [ ]:
with write_demo_notebook("08_edit_update.ipynb") as path:
    cell = mk_cell("x = 1")
    _write_tmp_nb(new_nb([cell]), path)
    session = ei.EditSession(path=path)
    edit_cell = ei.make_edit_tools(session)[1]
    result = edit_cell(cell.id, "2", "1")
    assert f"Edited text id={cell.id}" in result
    assert read_nb(path).cells[0].source == "x = 2"

In [ ]:
with write_demo_notebook("08_edit_missing.ipynb") as path:
    _write_tmp_nb(new_nb([mk_cell("x = 1")]), path)
    session = ei.EditSession(path=path)
    remove_cell = ei.make_edit_tools(session)[3]
    try:
        remove_cell("missing")
    except ValueError as exc:
        assert "No cell has id" in str(exc)
    else:
        raise AssertionError("expected missing cell failure")

In [ ]:
with write_demo_notebook("08_edit_run.ipynb") as path:
    c1 = mk_cell("x = 1")
    c2 = mk_cell("print(x + 1)")
    _write_tmp_nb(new_nb([c1, c2]), path)
    session = ei.EditSession(path=path, timeout=5)
    run_cell = ei.make_edit_tools(session)[2]
    text = run_cell(c2.id)
    assert "Executed" in text
    assert "2" in text

In [ ]:
class FakeChat:
    last = None

    def __init__(self, model, sp, tools, hist):
        self.model, self.sp, self.tools, self.hist = model, sp, tools, hist
        FakeChat.last = self

    def __call__(self, msg, max_steps=20, return_all=False):
        add_cell = self.tools[0]
        add_cell(None, "answer = 42")
        return "done; could not run extra checks"


from nbskill.foundation import write_demo_notebook

old_make_chat = ei.make_chat
try:
    ei.make_chat = lambda model, tools, hist, system_prompt=ei.EDIT_INTERACTIVE_SYSTEM: FakeChat(model, system_prompt, tools, hist)
    with write_demo_notebook("08_edit_plan.ipynb") as path:
        _write_tmp_nb(new_nb([mk_cell("#| default_exp sample")]), path)
        result = ei.execute_plan(str(path), "Add an answer cell.", model="fake")
        text = ei.plan_result_text(result)
        assert result["summary"] == "done; could not run extra checks"
        assert result["history"] and result["history"][0]["tool"] == "add_cell"
        assert "Final response:" in text
        assert "Tools used:" in text
        assert "add_cell" in text
        assert "done" in text
        assert "answer = 42" in _read_tmp_nb(path).cells[-1].source
        assert FakeChat.last.hist[1]["content"].count("answer = 42") == 1
finally:
    ei.make_chat = old_make_chat